# Лекция. Локализации, метод `apply` и конкатенации 

In [1]:
import numpy as np 
import pandas as pd

На этой лекции мы: 

- разберём разницу между выборкой по номеру `iloc` по метке `loc`, 
- поймём, как объединять несколько условий в один запрос и строить сложные булевы маски, 
- научимся генерировать новые признаки и сводные статистики при помощи универсального метода `apply`,
- узнаем, как объединять несколько датафреймов в один при помощи конкатенаций `concat`.

После чего
- сведем все эти приемы в разборе почти практической задачи Data Science.

## § 1. Метод `iloc`: выбор по номерам

Когда мы работаем с датафреймом, у него есть две системы координат:

- **Внешняя (видимая)**: имена столбцов и значения индекса строк (которые могут быть строками, датами, числами не по порядку и т. д.).
- **Внутренняя (скрытая)**: порядковые номера строк и столбцов, начинающиеся с нуля.


Метод `iloc` работает именно с внутренней системой. Ему всё равно, как называется столбец и какой индекс у строки — ему важен только порядковый номер. Другими словами, если вы создадите датафрейм с индексом [10, 20, 30], то `iloc[0]` вернёт строку с индексом 10, потому что `iloc` смотрит на номер позиции, а не на значение индекса.

Например, пусть у нас есть информация об успеваемости:

In [2]:
df = pd.DataFrame(np.random.randint(3, 6, size=(5, 7)),
                  index=['Маша', 'Катя', 'Петя', 'Женя', 'Лена'],
                  columns=['Математика', 'Физика', 'Информатика', 'ОБЖ', 'Литература', 'История', 'Английский'])
df

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,4,4,4,3,3,4
Катя,4,3,5,5,3,3,3
Петя,4,4,3,4,5,3,4
Женя,5,4,5,5,4,5,3
Лена,4,4,4,3,3,5,4


### Выбор строк по номерам

Строка с номером 0:

In [3]:
df.iloc[0]

Математика     3
Физика         4
Информатика    4
ОБЖ            4
Литература     3
История        3
Английский     4
Name: Маша, dtype: int32

Вообще, это оценки девочки Маши. Но это серия, что не всегда бывает удобно.

Строки с нулевой по вторую включительно (исключая третью) пулучаются при помощи среза:

In [4]:
df.iloc[0:3]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,4,4,4,3,3,4
Катя,4,3,5,5,3,3,3
Петя,4,4,3,4,5,3,4


Это уже датафрейм, что гораздо удобнее для восприятия.

Можно указать конкретный список номеров:

In [5]:
df.iloc[[3, 1, 1]]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Женя,5,4,5,5,4,5,3
Катя,4,3,5,5,3,3,3
Катя,4,3,5,5,3,3,3


Этот прикольный результат означает, что 1) строки будут идти в том порядке, который указан в списке, 2) строки могут повторяться столько раз, сколько соответствующий номер встретится в сиписке номеров.

А еще это означает, что мы можем схитрить и вывести одну строку как датафрейм (а не как серию). Для этого можно указать единственный номер не как число, а как список из одного элемента:

In [6]:
df.iloc[[0]]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,4,4,4,3,3,4


По другому того же эффекта можно добиться при помощи среза, у которого длина равна единице:

In [7]:
df.iloc[2:3]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Петя,4,4,3,4,5,3,4


### Выбор столбцов по номерам

В методе `iloc` столбцы указываются через запятую после строк. Можно обращаться к единственному столбцу, а можно — к нескольким, так же как и выше, при работе со строками.

Например, так вызывается конкретный единственный элемент:

In [8]:
df.iloc[2, 1]

4

Это элемент датафрейма, который находится на пересечении строки с номером 2 и столбца с номером 1 (или, если выражаться по-человечески, это оценка Кати по информатике). 

Аналогично можно получить прямоугольный фрагмент датафрейма, если использовать списки номеров (получится что-то наподобие минора в матрице):

In [9]:
df.iloc[[1, 2], [1, 3]]

,Физика,ОБЖ
Катя,3,5
Петя,4,4


Правда сходство с матрицами здесь весьма обманчиво, потому что, во-первых, число строк может отличаться от числа столбцов (а минор — это всегда квадратный фрагмент матрицы):

In [10]:
df.iloc[[1, 2], [1, 2, 3]]

,Физика,Информатика,ОБЖ
Катя,3,5,5
Петя,4,3,4


А во-вторых, в методе `iloc` можно менять порядок следования строк (или повторное использование и тех, и других):

In [11]:
df.iloc[[1, 1], [1, 1]]

,Физика,Физика
Катя,3,3
Катя,3,3


### Булевы маски в iloc

В качестве метода выбора строк можно использовать булевы маски. Например:

In [12]:
mask = df['Математика'] > 3
mask

Маша    False
Катя     True
Петя     True
Женя     True
Лена     True
Name: Математика, dtype: bool

Это серия. Но напрямую серию нельзя использовать в методе `iloc`. Поэтому мы извлекаем из нее массив при помощи метода `values` и применяем в качестве маски именно массив:

In [13]:
df.iloc[mask.values]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Катя,4,3,5,5,3,3,3
Петя,4,4,3,4,5,3,4
Женя,5,4,5,5,4,5,3
Лена,4,4,4,3,3,5,4


Наложить булеву маску можно и напрямую, без метода `iloc`:

In [14]:
df[mask.values]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Катя,4,3,5,5,3,3,3
Петя,4,4,3,4,5,3,4
Женя,5,4,5,5,4,5,3
Лена,4,4,4,3,3,5,4


И кстати, тогда можно использовать маску как серию, а не как массив:

In [15]:
df[mask]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Катя,4,3,5,5,3,3,3
Петя,4,4,3,4,5,3,4
Женя,5,4,5,5,4,5,3
Лена,4,4,4,3,3,5,4


Зачем тогда `iloc`? А затем, что без `iloc` возвращаются сразу все столбцы. Если же нужно локализовать по маске строки и вернуть не все столбцы, а только их часть, то без `iloc` не обойтись:

In [16]:
df.iloc[mask.values, 0:2]

,Математика,Физика
Катя,4,3
Петя,4,4
Женя,5,4
Лена,4,4


## § 2. Метод `loc`: выбор по меткам

Обращаться к эелементам датафрейма по номерам строк чаще всего бывает неудобно (потому что строк очень много). Обращаться по номерам столбцов — вообще нонсенс (так как они, как правило, именованы). Поэтому вместо выбора по номерам гораздо чаще используют выбор по меткам. 

Вернемся к тому же примеру:

In [17]:
df = pd.DataFrame(np.random.randint(3, 6, size=(5, 7)),
                  index=['Маша', 'Катя', 'Петя', 'Женя', 'Лена'],
                  columns=['Математика', 'Физика', 'Информатика', 'ОБЖ', 'Литература', 'История', 'Английский'])
df

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Катя,5,4,3,5,5,5,4
Петя,4,4,3,4,5,5,4
Женя,3,5,3,4,5,4,3
Лена,5,4,4,3,5,5,3


### Выбор строк по меткам

Оценки девочки Маши таковы:

In [18]:
df.loc['Маша']

Математика     3
Физика         5
Информатика    5
ОБЖ            4
Литература     4
История        5
Английский     5
Name: Маша, dtype: int32

Теперь выберем всех девочек (будем считать, что Женя — это мальчик):

In [19]:
df.loc[['Маша', 'Катя', 'Лена']]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Катя,5,4,3,5,5,5,4
Лена,5,4,4,3,5,5,3


В целом, все замечания, справедливые для выбора по номерам, справедливы и для выбора по меткам. Например, можно делать срезы:

In [20]:
df.loc['Катя':'Женя']

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Катя,5,4,3,5,5,5,4
Петя,4,4,3,4,5,5,4
Женя,3,5,3,4,5,4,3


Но есть и отличия. Дело в том, срезы в методе `loc` включают конечный элемент. Вспомним — в классических срезах правый край исключается, и если следовать этой логике, то во взятой выше локализации Жени не должно было быть. Но он есть. Такова специфика метода `loc`. 

В связи с этим, чтобы вывести значения одной отдельной строки в виде датафрейма (а не в виде серии), нужно применить срез нулевой длины:

In [21]:
df.loc['Катя':'Катя']

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Катя,5,4,3,5,5,5,4


### Выбор столбцов по меткам

Столбцы, как и строки, тоже можно выбирать по меткам. Например, можно получить доступ к отдельному элементу датафрейма:

In [22]:
df.loc['Маша', 'Физика']

5

Это оценка Маши по физике. Можно обратиться к меткам нескольких столбцов: 

In [23]:
df.loc['Маша', ['Физика', 'Английский']]

Физика        5
Английский    5
Name: Маша, dtype: int32

Получилась серия, что, вообще говоря, не очень удобно — ведь это фрагмент строки датафрейма. Можно схитрить и сделать единственную метку строк списком:

In [24]:
df.loc[['Маша'], ['Физика', 'Английский']]

,Физика,Английский
Маша,5,5


Так выглядит гораздо симпатичней.

И вообще, если указать два списка меток (сначало для строк, а потом — для столбцов), то получится прямоугольный фрагмент датафрейма:

In [25]:
df.loc[['Маша', 'Женя'], ['Физика', 'Английский']]

,Физика,Английский
Маша,5,5
Женя,5,3


По большому счету, все это не отличается от того, что мы говорили о выборе по номерам. Если, конечно, не считать правого края срезов.

### Булевы маски в `loc`

Сформируем маску по условию: оценки по физике должны быть отличными. Это серия:

In [26]:
mask = df['Физика'] == 5
mask

Маша     True
Катя    False
Петя    False
Женя     True
Лена    False
Name: Физика, dtype: bool

Хорошая новость в том, что теперь ее не обязательно переводить в массив. Локализация с серией в качестве маски тоже прекрасно сработает:

In [27]:
df.loc[mask]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Женя,3,5,3,4,5,4,3


Так же как и выше, в разделе о выборе по номерам, можно обойтись даже без метода `loc`:

In [28]:
df[mask]

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Женя,3,5,3,4,5,4,3


Но, во-первых, с помощью `loc` можно ограничить наше маскирующее условие на ограниченный набор стоолбцов, чего нельзя сделать без использования этого метода (обратите внимание, что оценки по физике вообще нет в выводе ячейки: маска задает условие, но не обязательно выводится):

In [29]:
df.loc[mask, ['Математика', 'Английский']]

,Математика,Английский
Маша,3,5
Женя,3,3


А во-вторых, использование `loc` универсально, поэтому удобнее всегда применять метод `loc`, чем держать в памяти ситуации, когда без него не обойтись, а когда можно и без него.

### Краткий обзор локализаций

Итак, отметим в качестве небольшого промежуточного саммари:
- метод выбора по меткам `loc` удобнее, чем метод выбора по номерам `iloc`,
- при применении `loc` правый край среза включается (в отличие от всех остальных ситуаций, в которых используются срезы),
- метод `loc` в качестве булевой маски позволяет использовать сразу серии (хотя можно использовать и массивы, и даже списки).

## § 3. Сложные маски и комбинирование условий

В классическом Python есть логические операторы `and`, `or` и `not`, которые работают с отдельными скалярными значениями True/False (мы применяли их раньше, когда строили циклы вместо векторных операций). В Pandas мы работаем с целыми сериями булевых значений. При этом 
```python 
and заменяется на &
``` 
```python 
or  заменяется на |
``` 
```python 
not заменяется на ~
```
и действует строгое правило: каждую операцию необходимо заключать в скобки. Поясним, почему это так на одном примере.

### Почему `and` не работает, а `&` работает, и почему нужны скобки

Вспомним, что у нас есть датафрейм с успеваемостью студентов:

In [30]:
df

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Катя,5,4,3,5,5,5,4
Петя,4,4,3,4,5,5,4
Женя,3,5,3,4,5,4,3
Лена,5,4,4,3,5,5,3


Потребуем, чтобы одновременно выполнялись 2 условия:
1. оценка по математике равна 4,
2. и при этом оценка по физике больше 3.

In [32]:
# (df['Математика'] == 4) and (df['Физика'] > 3)

Мы видим ошибку. А почему? Потому что оператор `and` — это логический оператор Python, работающий с одиночными булевыми значениями. При применении к серии pandas возникает неоднозначность: что считать «истинностью» всей серии? — и pandas выбрасывает ошибку.

Оператор `&` — это побитовый оператор (bitwise AND), который применяется поэлементно к двум сериям. Pandas переопределил его поведение для работы с булевыми сериями: он выполняет логическое `and` для каждой пары элементов (по сути, это все тот же цикл `for`, но снова выполненный под капотом pandas на быстрых компилируемых языках) и возвращает новую булеву серию той же длины, которую можно использовать для фильтрации.

Правильный вариант:

In [33]:
(df['Математика'] == 4) & (df['Физика'] > 3)

Маша    False
Катя    False
Петя     True
Женя    False
Лена    False
dtype: bool

Скобки критически важны из‑за приоритета операторов в Python. Без скобок Python сначала выполнит операцию `&`, а затем сравнение, потому что выражение без скобок
```python
df['Математика'] == 4 & df['Физика'] > 3
```
python, в силу приоритета логического оператора `&` над операторами сравнения `>` и `==`, интерпретирует как:
```python
df['Математика'] == (4 & df['Физика']) > 3
```
а это, разумеется, полный бред.

### Пример

Выведем оценки по литературе и истории всех студентов, у которых по математике, физике и информатике нет ни одной тройки:

In [34]:
df.loc[(df['Математика'] > 3) & (df['Физика'] > 3) & (df['Информатика'] > 3), ['Литература', 'История']]

,Литература,История
Лена,5,5


## § 4. Метод `apply`: продвинутые преобразования

В pandas есть замечательный метод: `apply`. Это метод, который позволяет применить произвольную функцию к столбцам и к строкам датафрейма. 

Вы скажете — ну и что? мэппинг тоже преобразовывает столбцы. Но во-первых, мэппинг не в состоянии реализовать любую функцию. Во-вторых, `applay`, в отличие от мэппинга, можно применять сразу к нескольким столбцам (другими словами, упомянутая выше любая функция может быть функцией нескольких переменных). А в третьих, `applay` можно применять не только к столбцам, но и к строкам (хотя, это используется гораздо реже). 

В общем, вполне себе универсальный метод всячески рекомендуемый к употреблению. Его базовый синтаксис:

```python
df.apply(function)          # по столбцам 
df.apply(function, axis=1)  # по строкам (axis=0 — значение по умолчанию)
```

А мэппинг (метод `map`) — это частный случай метода `apply` для одного столбца. Он «сопоставляет» каждое значение с каким-то другим значением по словарю. Главное правило:

- если работаем с одним столбцом и преобразование простое — можно использовать `map`.
- если нужно «пройтись строке» или применить сложную логику —  необходим `apply`.

### Пример, когда работает и `map` и `apply`

Берем наши данные с оценками:

In [35]:
df

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Катя,5,4,3,5,5,5,4
Петя,4,4,3,4,5,5,4
Женя,3,5,3,4,5,4,3
Лена,5,4,4,3,5,5,3


Допустим, мы хотим перевести численные оценки по математике в словесные: удовлетворительно, хорошо, отлично. Зачем нам это нужно — это другой вопрос, на самом деле, это в высшей степени дурацкая операция, так как обычно бывает нужно строго наоборот: строки переводить в числа. Ну вот захотелось нам так! Наши данные — что хотим, то и делаем.

Выводим очевидную логику соответствий во внешний словарь `math_dict` и применяем мэпинг к столбцу `df['Математика']`:

In [36]:
math_dict = {3:'удовл', 4: 'хор', 5: 'отл'} 
df['Математика'].map(math_dict)

Маша    удовл
Катя      отл
Петя      хор
Женя    удовл
Лена      отл
Name: Математика, dtype: object

То же самое можно сделать при помощи метода `apply`. Для этого логику перевода выносим во внешнюю функцию:

In [37]:
def word_score(row):
    if row['Математика'] == 3:
        return 'удовл'
    if row['Математика'] == 4:
        return 'хор'
    else:
        return 'отл'

После этого применяем метод apply ко всему датафрейму `df`, передав ему в качестве аргументов функцию `word_score` и направление вычислений `axis=1`:

In [38]:
df.apply(word_score, axis=1)

Маша    удовл
Катя      отл
Петя      хор
Женя    удовл
Лена      отл
dtype: object

Получается то же самое.

Казалось бы, зачем такие муки? Метод `map` гораздо логичнее и проще. Да, это так. Но это только пока вся наша логика сосредоточена внутри одного столбца. Если же она захватывает несколько столбцов, то метод `map` с такой логикой уже не справится.

### Пример, когда `map` не поможет, только `apply`

Допустим, я хочу создать столбец 'Отношение' со сленговым обозначением моего отношения к успехам того или иного студента. Критерии, в соответствии с которыми формируется отношение, заложены в его оценках по математике A и информатике B. А именно:

1) если A = 5 и B = 5 или A = 4 и B = 5 или A = 5 и B = 4, то мое отношение к этому персонажу — 'краш',

2) если A = 4 и B = 4 или A = 4 и B = 3 или A = 3 и B = 4, то  отношение — 'норм',

3) во всех остальных случаях — 'кринж'.

Выносим мою логику во внешнюю функцию `sleng`:

In [39]:
def sleng(row):
    A = row['Математика']
    B = row['Информатика']

    if (A == 5 and B == 5) or (A == 4 and B == 5) or (A == 5 and B == 4):
        return 'краш'
    elif (A == 4 and B == 4) or (A == 4 and B == 3) or (A == 3 and B == 4):
        return 'норм'
    else:
        return 'кринж'

А потом применяем к датафрейму метод `apply` с функцией `sleng`, причем, делаем это построчно (указываем `axis=1`):

In [40]:
df.apply(sleng, axis=1)

Маша    кринж
Катя    кринж
Петя     норм
Женя    кринж
Лена     краш
dtype: object

В результате получается серия. 

Чуть подробнее о том, как `apply` работает под капотом:

```python
Шаг 1: Берём строку 0 (Маша)
        ↓
        sleng( row = Series([4, 5, 3, 5, 4, 4, 5], # последовательность чисел может быть другой
                            index=['Математика','Физика',...,'Английский']) ) # информатика на третьем месте
        ↓
        возвращаем 'норм' # в зависимости от оценок

Шаг 2: Берём строку 1 (Катя)
        ↓
        sleng( row = Series([5, 5, 4, 3, 4, 5, 4], 
                            index=['Математика','Физика',...,'Английский']) )
        ↓
        возвращаем 'краш'

...и так далее до конца таблицы
```
Ключевые моменты всей этой процедуры:
- переменная `row` — это одна строка датафрейма `df`, переданная в функцию `sleng` как серия,
- индекс этой серии — названия столбцов,
- `apply` перебирает строки сверху вниз (это еще один пример векторной работы, которая, по сути является циклом `for`, только выполняется очень быстро),
- каждый результат складывается в выходную серию, которую потом можно сделать новым столбцом (хотя можно и не делать).

In [41]:
df['Отношение'] = df.apply(sleng, axis=1)
df

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский,Отношение
Маша,3,5,5,4,4,5,5,кринж
Катя,5,4,3,5,5,5,4,кринж
Петя,4,4,3,4,5,5,4,норм
Женя,3,5,3,4,5,4,3,кринж
Лена,5,4,4,3,5,5,3,краш


Этот столбец нам больше не понадобится, поэтому мы его удалим:

In [42]:
df = df.drop(columns = ['Отношение'])
df

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Катя,5,4,3,5,5,5,4
Петя,4,4,3,4,5,5,4
Женя,3,5,3,4,5,4,3
Лена,5,4,4,3,5,5,3


### Пример со сменой логики: `axis=0`

Допустим, я хочу выяснить, как студенты Маша и Петя относятся к предметам, которые они изучают. Со словарным запасом у них не очень, поэтому слова они используют все те же: краш, норм и кринж — и принципы оценки того или иного предмета остаются теми же, что и выше.

Формулируем логику Маши и Пети во внешенй функции `sleng` (обратите внимание, что я ее практически не изменяю):

In [44]:
def sleng(column):
    A = column['Маша']
    B = column['Петя']

    if (A == 5 and B == 5) or (A == 4 and B == 5) or (A == 5 and B == 4):
        return 'краш'
    elif (A == 4 and B == 4) or (A == 4 and B == 3) or (A == 3 and B == 4):
        return 'норм'
    else:
        return 'кринж'

И передаем функцию `sleng` в метод `apply` в качестве аргумента, вообще не указывая значение `axis` (потому что `axis=0` — это дефолтное значение):

In [45]:
df.apply(sleng)

Математика      норм
Физика          краш
Информатика    кринж
ОБЖ             норм
Литература      краш
История         краш
Английский      краш
dtype: object

В результате получается серия. 

Как на этот раз работает  `apply`:

```python
Шаг 1: Берём столбец 0 (Математика)
        ↓
        sleng( row = Series([4, 5, 3, 5, 4], # последовательность чисел может быть другой
                            index=['Маша', 'Катя', 'Петя', 'Женя', 'Лена']) ) # Нам нужны только Маша и Петя
        ↓
        возвращаем 'норм' # в зависимости от оценок

Шаг 2: Берём строку 1 (Физика)
        ↓
        sleng( row = Series([5, 5, 4, 3, 4, 5, 4], 
                            index=['Маша', 'Катя', 'Петя', 'Женя', 'Лена']) )
        ↓
        возвращаем 'краш'

...и так далее до конца таблицы
```
Ключевые моменты всей этой процедуры:
- переменная `column` — это один столбец, переданный в функцию `sleng` как серия,
- индекс этой серии — названия строк,
- `apply` перебирает столбцы слева направо,
- каждый результат складывается в выходную серию, которую потом можно сделать новой строкой (хотя можно и не делать).

### Почему `axis = 0` по дефлоту?

В двух словах — так исторически сложилось.

Это наследие, доставшееся от классической математической статистики, в которой принято считать, что :

- строки — это наблюдения (объекты),
- столбцы — признаки.

И когда мы говорим «посчитать среднее», мы обычно имеем в виду среднее по столбцам — то есть, что для каждого признака нам нужно посчитать его среднее значение по всем наблюдениям. Результат — это одна строка со средними значениями, то есть мы как бы конвертируем все строки в одну, содержащую сводную статистику.

В примере выше мы тоже создали некую сводную статистику данных: вычислили отношение Маши и Пети ко всем предметам, которые они изучают. С одной стороны, какая же это статистика? — ведь мы учли мнение только двух студентов (а не всех сразу). Но с другой стороны, в этом и заключается сила метода `applay`: мы можем формировать любые статистики. Вообще любые.

Вывод:

- `axis=1` — это про создание новых признаков, что чаще используется на этапе построения моделей (feature engineering), 
- `axis=0` — это про понимание существующих признаков, что чаще используется на этапе предварительного анализа данных (summary statistics).

## § 5. Конкатенации

В старых версиях Pandas был метод `append`, который добавлял строку в конец датафрейма по аналогии с тем, как это делается со списками. Но он всех раздражал, потому что работал медленно и не поддерживал добавление столбцов. Поэтому, начиная с Pandas 2.0, метод `append` объявлен устаревшим и удалён. Вместо него теперь — метод `concat`. 

Общий синтаксис метода таков:
```python
pd.concat([df1, df2])           # добавляются строки сверху вниз
pd.concat([df1, df2], axis=1)   # добавляются столбцы слева направо
```

### Пример добавления строк

Допустим, в нашем весёлом фрик-шоу появилось еще два ученика:

In [46]:
df_new_students = pd.DataFrame(np.random.randint(3, 6, size=(2, 7)),
                  index=['Соня', 'Валера'],
                  columns=['Математика', 'Физика', 'Информатика', 'ОБЖ', 'Литература', 'История', 'Английский'])
df_new_students

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Соня,5,4,3,4,4,5,3
Валера,4,5,5,4,3,5,3


Добавить их в общий список проще простого:

In [47]:
df_combined = pd.concat([df, df_new_students]) 
df_combined

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Катя,5,4,3,5,5,5,4
Петя,4,4,3,4,5,5,4
Женя,3,5,3,4,5,4,3
Лена,5,4,4,3,5,5,3
Соня,5,4,3,4,4,5,3
Валера,4,5,5,4,3,5,3


Но что будет, если вновом списке окажется еще одна Маша?

In [48]:
df_new_students = pd.DataFrame(np.random.randint(3, 6, size=(3, 7)),
                  index=['Соня', 'Валера', 'Маша'],
                  columns=['Математика', 'Физика', 'Информатика', 'ОБЖ', 'Литература', 'История', 'Английский'])
df_new_students

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Соня,4,3,3,4,4,4,5
Валера,4,4,4,3,4,3,4
Маша,3,5,4,4,5,4,5


Тогда после конкатенации получится две Маши. И как их отличать друг от друга?

In [49]:
df_combined = pd.concat([df, df_new_students]) 
df_combined

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Катя,5,4,3,5,5,5,4
Петя,4,4,3,4,5,5,4
Женя,3,5,3,4,5,4,3
Лена,5,4,4,3,5,5,3
Соня,4,3,3,4,4,4,5
Валера,4,4,4,3,4,3,4
Маша,3,5,4,4,5,4,5


Можно поступить интеллигентно и использовать параметр `keys`, который добавляет уще один уровень индекса (то есть, создает мультииндекс):

In [50]:
df_combined_with_names = pd.concat([df, df_new_students], keys=['old', 'new'])
df_combined_with_names

Математика  Физика  Информатика  ОБЖ  Литература  История  \
old Маша             3       5            5    4           4        5   
    Катя             5       4            3    5           5        5   
    Петя             4       4            3    4           5        5   
    Женя             3       5            3    4           5        4   
    Лена             5       4            4    3           5        5   
new Соня             4       3            3    4           4        4   
    Валера           4       4            4    3           4        3   
    Маша             3       5            4    4           5        4   

            Английский  
old Маша             5  
    Катя             4  
    Петя             4  
    Женя             3  
    Лена             3  
new Соня             5  
    Валера           4  
    Маша             5

Теперь старая Маша — это не то же самое, что новая Маша. Обращаться к ним можно при помощи метода `loc` за счет комбинированного индекса (обратите внимание, что в качестве индекса используется кортеж):

In [51]:
display(df_combined_with_names.loc[[('old', 'Маша')]])
display(df_combined_with_names.loc[[('new', 'Маша')]])

,,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
old,Маша,3,5,5,4,4,5,5


,,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
new,Маша,3,5,4,4,5,4,5


Однако, чаще всего в задачах машинного обучения просто сбрасывают индексы за счет параметра `ignore_index`:

In [52]:
df_combined_without_names = pd.concat([df, df_new_students], ignore_index=True)
df_combined_without_names

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
0,3,5,5,4,4,5,5
1,5,4,3,5,5,5,4
2,4,4,3,4,5,5,4
3,3,5,3,4,5,4,3
4,5,4,4,3,5,5,3
5,4,3,3,4,4,4,5
6,4,4,4,3,4,3,4
7,3,5,4,4,5,4,5


Мы при этом, конечно, теряем имена учеников, но такие детали редко кого интересуют при анализе больших массивов данных. А если вам так нужны имена, то используйте их в качестве отдельного признака, а не в качестве индекса. Пусть индекс будет числовым и уникальным счетчиком.

Представим другую ситуацию. Допустим, к нам приходят новые ученики, но у ниж не было ОБЖ:

In [53]:
display(df)
df_new_students = pd.DataFrame(np.random.randint(3, 6, size=(3, 6)),
                  index=['Соня', 'Валера', 'Маша'],
                  columns=['Математика', 'Физика', 'Информатика', 'Литература', 'История', 'Английский'])
df_new_students

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4,4,5,5
Катя,5,4,3,5,5,5,4
Петя,4,4,3,4,5,5,4
Женя,3,5,3,4,5,4,3
Лена,5,4,4,3,5,5,3


,Математика,Физика,Информатика,Литература,История,Английский
Соня,3,3,5,3,4,4
Валера,3,4,4,5,5,5
Маша,3,4,5,3,3,4


Объединяем их в один датафрейм:

In [54]:
pd.concat([df, df_new_students]) 

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский
Маша,3,5,5,4.0,4,5,5
Катя,5,4,3,5.0,5,5,4
Петя,4,4,3,4.0,5,5,4
Женя,3,5,3,4.0,5,4,3
Лена,5,4,4,3.0,5,5,3
Соня,3,3,5,NaN,3,4,4
Валера,3,4,4,NaN,5,5,5
Маша,3,4,5,NaN,3,3,4


Получилось замечательно. Умничка pandas поставил NaN'ы вместо отсутствующих оценок (обратите внимание, что оценки стали дробными — мы понимаем, что это из-за NaN'ов). Если нам не нужен такой ущербный столбец, мы можем избавиться от него при помощи метода `drop`:

In [55]:
df_combine = pd.concat([df, df_new_students]) 
df_combine = df_combine.drop(columns = ['ОБЖ'])
df_combine

,Математика,Физика,Информатика,Литература,История,Английский
Маша,3,5,5,4,5,5
Катя,5,4,3,5,5,4
Петя,4,4,3,5,5,4
Женя,3,5,3,5,4,3
Лена,5,4,4,5,5,3
Соня,3,3,5,3,4,4
Валера,3,4,4,5,5,5
Маша,3,4,5,3,3,4


Но можем поступить изящнее: не учитывать столбцы, которые не являются общими для обоих датафреймов заранее (еще на этапе их слияния) за счет атрибута `join`:

In [56]:
df_combine = pd.concat([df, df_new_students], join='inner') 
df_combine

,Математика,Физика,Информатика,Литература,История,Английский
Маша,3,5,5,4,5,5
Катя,5,4,3,5,5,4
Петя,4,4,3,5,5,4
Женя,3,5,3,5,4,3
Лена,5,4,4,5,5,3
Соня,3,3,5,3,4,4
Валера,3,4,4,5,5,5
Маша,3,4,5,3,3,4


Дефолтное значение этого атрибута — `'outer'` (при этом учитываются все столбцы).

### Пример добавления столбцов

Пусть теперь мы получили дополнительные сведения о наших учениках. Теперь мы знаем их оценки по географии и ИЗО:

In [57]:
df_new_subject = pd.DataFrame(np.random.randint(3, 6, size=(5, 2)),
                  index=['Маша', 'Катя', 'Петя', 'Женя', 'Лена'],
                  columns=['География', 'ИЗО'])
df_new_subject

,География,ИЗО
Маша,5,3
Катя,3,3
Петя,4,5
Женя,5,3
Лена,3,4


Приписать новые сведения справа от имеющихся очень легко при помощи метода `concat` с указанием атрибута `axis=1`:

In [58]:
df_combine = pd.concat([df, df_new_subject], axis=1)
df_combine

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский,География,ИЗО
Маша,3,5,5,4,4,5,5,5,3
Катя,5,4,3,5,5,5,4,3,3
Петя,4,4,3,4,5,5,4,4,5
Женя,3,5,3,4,5,4,3,5,3
Лена,5,4,4,3,5,5,3,3,4


Если же, например, Петя не изучал эти предметы, то мы естественно получим NaN'ы:

In [59]:
df_new_subject = pd.DataFrame(np.random.randint(3, 6, size=(4, 2)),
                  index=['Маша', 'Катя', 'Женя', 'Лена'],
                  columns=['География', 'ИЗО'])
display(df_new_subject)

df_combine = pd.concat([df, df_new_subject], axis=1)
df_combine

,География,ИЗО
Маша,4,3
Катя,3,3
Женя,5,4
Лена,5,5


,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский,География,ИЗО
Маша,3,5,5,4,4,5,5,4.0,3.0
Катя,5,4,3,5,5,5,4,3.0,3.0
Петя,4,4,3,4,5,5,4,NaN,NaN
Женя,3,5,3,4,5,4,3,5.0,4.0
Лена,5,4,4,3,5,5,3,5.0,5.0


Мы можем легко избавиться от этоя строки (но как избавиться от Пети?) при помощи метода `dropna`. Но можем сделать это еще на этапе слияния, за счет атрибута `join='inner'`:

In [60]:
df_combine = pd.concat([df, df_new_subject], axis=1, join = 'inner')
df_combine

,Математика,Физика,Информатика,ОБЖ,Литература,История,Английский,География,ИЗО
Маша,3,5,5,4,4,5,5,4,3
Катя,5,4,3,5,5,5,4,3,3
Женя,3,5,3,4,5,4,3,5,4
Лена,5,4,4,3,5,5,3,5,5


Хотя, конечно, Петю жалко. И вообще, что делать с возникающими пропусками — это отдельная тема, кторая требует детального исследования в каждом конкретном случае (нельзя же, действительно, людей из жизни вычеркивать).

### Краткий обзор конкатенаций

Резюмируем сказанное в виде небольшого саммари:
- метод `append` для датафреймов, увы, приказал долго жить, поэтому всегда используем `concat`,
- по умолчанию конкатенация осуществляется по строкам (сверху вниз),
- при указании `axis=1` конкатенация идет по столбцам (слева направо),
- если нужно сбросить индекс, то используем атрибут `ignore_index=True`,
- если в контатенируемых датафреймах есть разные столбцы (или строки), то появятся NaN'ы,
- чтобы подавить NaN'ы используем атрибут `join='inner'`(по умолчанию `join='outer'`).

## § 6. Интеграция в Data Science

Допустим, у нас есть 2 класса по 25 человек в каждом: класс А и класс B. 

В 2024 году дети изучали: Алгебру, Геометрию, Информатику, Физику — это по естественнонаучному блоку. А кроме того: Историю, Литературу, ИЗО и Английский — это по гуманитарному блоку. В 2025 году они перешли в следующий класс и изучали те же предметы, кроме ИЗО (потому что они выросли, и у них больше нет рисования).

Вот эти данные (мы вывели по 3 строки, но вообще их там по 50 в каждом датафрейме):

In [61]:
df_2024 = pd.read_csv('data/school_score_2024.csv')
df_2025 = pd.read_csv('data/school_score_2025.csv')
display(df_2024.head(3))
display(df_2025.head(3))

,ID,Алгебра,Геометрия,Информатика,Физика,История,Литература,ИЗО,Английский
0,A13,3,5,3,3,5,5,5,3
1,A07,4,4,4,3,4,3,4,5
2,B13,4,4,4,4,5,3,3,3


,ID,Алгебра,Геометрия,Информатика,Физика,История,Литература,Английский
0,B19,5,3,3,4,4,5,5
1,B14,3,5,3,5,3,5,3
2,B10,4,5,3,3,4,3,5


Мы понимаем, что скоро нам предстоит распределять наших учеников по двум классам: тех что поумнее мы отправим в так называемый математический класс, а совсем тупых — в гуманитарный. И мы хотим выяснить, каково предварительное распределение на текущий момент.

Логика нашего распределения будет незамысловата: если средняя оценка ученика по естественнонаучному блоку больше средней оценки по гуманитарному блоку, то ребенок идет в математический класс. Иначе — в гуманитарный. 

Выводим нашу логику во внешнюю функцию `is_clever`:

In [62]:
def is_clever(row):
    if row[1:5].mean() > row[5:].mean():
        return 'Математический'
    else:
        return 'Гуманитарный'

Далее, при помощи метода `apply` применяем нашу логику сначала к 2024 году, потом — к 2025. Обратите внимание, что мы используем атрибут `axis=1`, потому что действуем в пределах каждой строки:

In [63]:
mind_2024 = df_2024.apply(is_clever, axis=1)
mind_2025 = df_2025.apply(is_clever, axis=1)

display(mind_2024.head(3))
display(mind_2025.head(3))

0      Гуманитарный
1      Гуманитарный
2    Математический
dtype: object

0      Гуманитарный
1    Математический
2      Гуманитарный
dtype: object

Получаем две серии. Мы вывели по 3 записи в каждой из них, но вообще их там по 50 в каждой серии.

Дальше нас интересует такой вопрос: кто из учеников за год обучения изменил свою направленность, и у кого она осталась неизменной? И если направленность изменилась, то как? Если с гуманитарной на математическую, то это хорошо, а если наоборот — плохо.

Мы не можем напрямую сравнить две серии, полученные выше, потому что ученики в них следуют в разных порядках. Нам нужно использовать в качестве индексов этих серий ID учеников (тогда возникнет взаимосвязь между двумя сериями).

In [64]:
mind_2024.index = df_2024['ID']
mind_2025.index = df_2025['ID']

display(mind_2024.head(3))
display(mind_2025.head(3))

ID
A13      Гуманитарный
A07      Гуманитарный
B13    Математический
dtype: object

ID
B19      Гуманитарный
B14    Математический
B10      Гуманитарный
dtype: object

Конкатенируем эти две серии (да-да, конкатенировать можно не только датафреймы, но и серии), прикрепляя 2025 год справа к 2024:

In [65]:
mind_combine = pd.concat([mind_2024, mind_2025], axis=1)
mind_combine.head(3)

,0,1
ID,,
A13,Гуманитарный,Гуманитарный
A07,Гуманитарный,Гуманитарный
B13,Математический,Гуманитарный


Тперь снова включаем логику метода `apply`:

In [66]:
def change(row):
    if row[0] == 'Гуманитарный' and row[1] == 'Математический':
        return '↑'
    elif row[0] == 'Математический' and row[1] == 'Гуманитарный':
        return '↓'
    else:
        return '—'

И после применения метода `apply` с атрибутом `axis=1`, получаем сведения о прогрессе, деградации или неизменности наших учеников:

In [67]:
progres = mind_combine.apply(change, axis=1)
progres.head(5)

ID
A13    —
A07    —
B13    ↓
A09    ↑
A21    ↓
dtype: object

Теперь при помощи метода `loc` мы можем вывести всех деградировавших учеников и провести с ними воспитательную беседу.

In [68]:
progres.loc[progres == '↓']

ID
B13    ↓
A21    ↓
A08    ↓
A06    ↓
B24    ↓
A20    ↓
B16    ↓
A03    ↓
A16    ↓
B12    ↓
B11    ↓
A19    ↓
A02    ↓
B22    ↓
dtype: object

# Домашнее задание

Итак, у нас есть 50 учеников, и есть сведения об их успеваемости за 2 года (см. выше). 

- Объедините эти сведения в общую таблицу при помощи метода `concat`,
- получите при помощи метода `apply` сводную статистику за два года: по каждому предмету мы хотим знать дельту, то есть величину, на которую изменилась средняя успеваемость в 2025 году по сравнению с 2024,
- объедините таблицу успеваемости со сводной статистикой, добавив статистику последней строкой.